In [ ]:
# ============================================================
# MITT THANDAVAPURA COLLEGE Q&A ASSISTANT
# PAGE-BY-PAGE + BLACK THEME + COLOURFUL UI
#
# Pages:
# 1. Home
# 2. College Details
# 3. Load Website
# 4. Suggested Questions
# 5. Ask Question
# 6. About Project
#
# Technologies:
# - Python
# - Requests
# - BeautifulSoup
# - Gradio
# - Keyword-based search
#
# No:
# - Qwen
# - FAISS
# - Embeddings
# - Vector database
# - Chatbot component
# ============================================================


# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip -q install requests beautifulsoup4 gradio


# ============================================================
# 2. IMPORTS
# ============================================================

import requests
import re
import time
import gradio as gr

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from collections import deque


# ============================================================
# 3. COLLEGE CONFIGURATION
# ============================================================

COLLEGE_NAME = "Maharaja Institute of Technology Thandavapura"

COLLEGE_SHORT_NAME = "MITT"

COLLEGE_URL = "https://mitt.edu.in/"

BACKGROUND_IMAGE = (
    "https://gopalaswamyinstitutions.in/"
    "wp-content/uploads/2024/03/MITT-Campus.jpg"
)


# ============================================================
# 4. SETTINGS
# ============================================================

MAX_PAGES = 25

REQUEST_TIMEOUT = 12

MIN_TEXT_LENGTH = 80

MAX_CHUNKS_PER_PAGE = 25

CHUNK_WORDS = 120


# ============================================================
# 5. GLOBAL VARIABLES
# ============================================================

website_loaded = False

college_pages = []

website_content = []

visited_urls = set()

base_domain = ""


# ============================================================
# 6. HTTP SESSION
# ============================================================

session = requests.Session()

session.headers.update({

    "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36",

    "Accept":
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8",

    "Accept-Language":
        "en-US,en;q=0.9"
})


# ============================================================
# 7. NORMALIZE URL
# ============================================================

def normalize_url(url):

    if not url:
        return ""

    url = str(url).strip()

    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    try:

        parsed = urlparse(url)

        scheme = parsed.scheme.lower()

        hostname = parsed.hostname

        if not hostname:
            return ""

        hostname = hostname.lower()

        if hostname.startswith("www."):
            hostname = hostname[4:]

        path = parsed.path or "/"

        path = re.sub(r"/+", "/", path)

        if path != "/" and path.endswith("/"):
            path = path[:-1]

        return (
            scheme
            + "://"
            + hostname
            + path
        )

    except Exception:

        return ""


# ============================================================
# 8. GET DOMAIN
# ============================================================

def get_domain(url):

    try:

        hostname = urlparse(url).hostname

        if not hostname:
            return ""

        hostname = hostname.lower()

        if hostname.startswith("www."):
            hostname = hostname[4:]

        return hostname

    except Exception:

        return ""


# ============================================================
# 9. SAME DOMAIN
# ============================================================

def same_domain(url):

    domain = get_domain(url)

    if not domain:
        return False

    if not base_domain:
        return False

    return (
        domain == base_domain
        or domain.endswith("." + base_domain)
    )


# ============================================================
# 10. VALID URL
# ============================================================

def valid_url(url):

    if not url:
        return False

    try:

        parsed = urlparse(url)

        if parsed.scheme not in [
            "http",
            "https"
        ]:
            return False

        if not parsed.netloc:
            return False

        path = parsed.path.lower()

        blocked_extensions = (

            ".jpg",
            ".jpeg",
            ".png",
            ".gif",
            ".webp",
            ".svg",
            ".ico",

            ".mp3",
            ".wav",

            ".mp4",
            ".avi",
            ".mov",
            ".mkv",

            ".zip",
            ".rar",
            ".7z",

            ".css",
            ".js",

            ".woff",
            ".woff2",
            ".ttf",

            ".xlsx",
            ".xls",
            ".doc",
            ".docx",
            ".ppt",
            ".pptx",
            ".pdf"
        )

        if path.endswith(blocked_extensions):
            return False

        return True

    except Exception:

        return False


# ============================================================
# 11. CLEAN TEXT
# ============================================================

def clean_text(text):

    if not text:
        return ""

    text = text.replace("\xa0", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# 12. DOWNLOAD PAGE
# ============================================================

def download_page(url):

    try:

        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True
        )

        print(
            f"[{response.status_code}] {url}"
        )

        if response.status_code != 200:
            return None

        content_type = response.headers.get(
            "Content-Type",
            ""
        ).lower()

        if "text/html" not in content_type:
            return None

        if len(response.content) == 0:
            return None

        return response

    except requests.exceptions.Timeout:

        print("TIMEOUT:", url)

        return None

    except requests.exceptions.RequestException as e:

        print("REQUEST ERROR:", str(e))

        return None

    except Exception as e:

        print("DOWNLOAD ERROR:", str(e))

        return None


# ============================================================
# 13. EXTRACT PAGE
# ============================================================

def extract_page(url, response):

    try:

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup.find_all([
            "script",
            "style",
            "noscript",
            "svg",
            "canvas",
            "iframe",
            "form",
            "nav",
            "footer",
            "header"
        ]):

            tag.decompose()

        title = ""

        if soup.title:

            title = soup.title.get_text(
                " ",
                strip=True
            )

        main = soup.find("main")

        if main:

            text = main.get_text(
                " ",
                strip=True
            )

        else:

            body = soup.find("body")

            if body:

                text = body.get_text(
                    " ",
                    strip=True
                )

            else:

                text = soup.get_text(
                    " ",
                    strip=True
                )

        text = clean_text(text)

        if len(text) < MIN_TEXT_LENGTH:
            return None

        return {

            "url": url,
            "title": title,
            "text": text

        }

    except Exception as e:

        print(
            "EXTRACTION ERROR:",
            str(e)
        )

        return None


# ============================================================
# 14. EXTRACT LINKS
# ============================================================

def extract_links(page_url, response):

    links = []

    try:

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup.find_all("a"):

            href = tag.get("href")

            if not href:
                continue

            href = href.strip()

            if href.startswith((
                "#",
                "mailto:",
                "tel:",
                "javascript:",
                "whatsapp:"
            )):
                continue

            absolute = urljoin(
                page_url,
                href
            )

            absolute = urldefrag(
                absolute
            )[0]

            absolute = normalize_url(
                absolute
            )

            if not absolute:
                continue

            if not same_domain(absolute):
                continue

            if not valid_url(absolute):
                continue

            links.append(absolute)

    except Exception as e:

        print(
            "LINK ERROR:",
            str(e)
        )

    return list(
        dict.fromkeys(links)
    )


# ============================================================
# 15. CRAWL WEBSITE
# ============================================================

def crawl_website():

    global base_domain
    global college_pages
    global visited_urls

    college_pages = []

    visited_urls = set()

    start_url = normalize_url(
        COLLEGE_URL
    )

    base_domain = get_domain(
        start_url
    )

    queue = deque()

    queue.append(
        start_url
    )

    print()
    print("=" * 70)
    print("STARTING MITT WEBSITE CRAWLER")
    print("=" * 70)

    while (
        queue
        and len(college_pages) < MAX_PAGES
    ):

        current_url = queue.popleft()

        current_url = normalize_url(
            current_url
        )

        if not current_url:
            continue

        if current_url in visited_urls:
            continue

        if not same_domain(current_url):
            continue

        if not valid_url(current_url):
            continue

        visited_urls.add(
            current_url
        )

        print(
            f"\n[{len(college_pages)+1}/{MAX_PAGES}] "
            f"{current_url}"
        )

        response = download_page(
            current_url
        )

        if response is None:
            continue

        final_url = normalize_url(
            response.url
        )

        if final_url:

            current_url = final_url

        page = extract_page(
            current_url,
            response
        )

        if page:

            college_pages.append(
                page
            )

            print(
                "   ✓ Saved:",
                page["title"]
            )

        links = extract_links(
            current_url,
            response
        )

        for link in links:

            if link in visited_urls:
                continue

            if link in queue:
                continue

            if len(queue) >= MAX_PAGES * 3:
                break

            queue.append(link)

        time.sleep(0.05)

    print()
    print("=" * 70)
    print("CRAWLING FINISHED")
    print("=" * 70)

    print(
        "Pages loaded:",
        len(college_pages)
    )

    print(
        "URLs visited:",
        len(visited_urls)
    )

    return college_pages


# ============================================================
# 16. BUILD SEARCH CONTENT
# ============================================================

def build_search_content():

    global website_content

    website_content = []

    for page in college_pages:

        text = page["text"]

        words = text.split()

        start = 0

        chunk_count = 0

        while (
            start < len(words)
            and chunk_count < MAX_CHUNKS_PER_PAGE
        ):

            chunk_words = words[
                start:start + CHUNK_WORDS
            ]

            chunk = " ".join(
                chunk_words
            )

            if len(chunk) >= 80:

                website_content.append({

                    "text": chunk,

                    "url": page["url"],

                    "title": page["title"]

                })

                chunk_count += 1

            start += CHUNK_WORDS

    print(
        "Search chunks:",
        len(website_content)
    )


# ============================================================
# 17. QUESTION TOPICS
# ============================================================

QUESTION_TOPICS = {

    "courses": [
        "course",
        "courses",
        "program",
        "programs",
        "degree",
        "degrees",
        "undergraduate",
        "postgraduate",
        "btech",
        "b.e",
        "be",
        "mtech",
        "mba",
        "mca",
        "phd",
        "engineering"
    ],

    "admission": [
        "admission",
        "admissions",
        "apply",
        "application",
        "enrolment",
        "enrollment"
    ],

    "eligibility": [
        "eligibility",
        "eligible",
        "qualification",
        "qualifications",
        "criteria",
        "requirements"
    ],

    "fees": [
        "fee",
        "fees",
        "tuition",
        "cost",
        "fee structure"
    ],

    "placements": [
        "placement",
        "placements",
        "recruiter",
        "recruiters",
        "company",
        "companies",
        "career",
        "careers",
        "salary",
        "package"
    ],

    "hostel": [
        "hostel",
        "hostels",
        "accommodation",
        "residence",
        "room"
    ],

    "scholarship": [
        "scholarship",
        "scholarships",
        "financial aid",
        "funding"
    ],

    "departments": [
        "department",
        "departments",
        "school",
        "schools",
        "faculty"
    ],

    "campus": [
        "campus",
        "campuses",
        "infrastructure",
        "facility",
        "facilities"
    ],

    "library": [
        "library",
        "libraries",
        "books",
        "reading"
    ],

    "sports": [
        "sports",
        "sport",
        "playground",
        "gym",
        "fitness"
    ],

    "clubs": [
        "club",
        "clubs",
        "student club",
        "student clubs",
        "activities",
        "events"
    ],

    "internship": [
        "internship",
        "internships",
        "training",
        "industrial training"
    ],

    "research": [
        "research",
        "research center",
        "research centre",
        "innovation",
        "phd"
    ],

    "location": [
        "location",
        "located",
        "address",
        "campus address",
        "where is",
        "situated"
    ],

    "about": [
        "about",
        "history",
        "established",
        "vision",
        "mission",
        "known"
    ],

    "facilities": [
        "facility",
        "facilities",
        "laboratory",
        "laboratories",
        "lab",
        "labs",
        "transport",
        "transportation",
        "infrastructure"
    ]

}


# ============================================================
# 18. DETECT TOPIC
# ============================================================

def detect_topic(question):

    q = question.lower()

    scores = {}

    for topic, keywords in QUESTION_TOPICS.items():

        score = 0

        for keyword in keywords:

            if keyword in q:
                score += 1

        scores[topic] = score

    best_topic = max(
        scores,
        key=scores.get
    )

    if scores[best_topic] == 0:

        return "general"

    return best_topic


# ============================================================
# 19. SEARCH CONTENT
# ============================================================

def search_content(question):

    if not website_content:
        return []

    q = question.lower()

    topic = detect_topic(q)

    words = re.findall(
        r"[a-zA-Z]{3,}",
        q
    )

    stop_words = {

        "what",
        "which",
        "where",
        "when",
        "who",
        "how",
        "does",
        "are",
        "is",
        "the",
        "this",
        "that",
        "college",
        "university",
        "please",
        "tell",
        "about",
        "give",
        "me",
        "can",
        "you",
        "for",
        "from",
        "with",
        "have",
        "has",
        "their",
        "there",
        "available",
        "offer",
        "offers",
        "provide",
        "provides"
    }

    keywords = [
        word
        for word in words
        if word not in stop_words
    ]

    topic_keywords = QUESTION_TOPICS.get(
        topic,
        []
    )

    scored = []

    for item in website_content:

        text = item["text"].lower()

        score = 0

        for keyword in keywords:

            if keyword in text:
                score += 2

        for keyword in topic_keywords:

            if keyword in text:
                score += 5

        title = item["title"].lower()

        for keyword in topic_keywords:

            if keyword in title:
                score += 8

        if score > 0:

            scored.append(
                (
                    score,
                    item
                )
            )

    scored.sort(
        key=lambda x: x[0],
        reverse=True
    )

    results = []

    seen = set()

    for score, item in scored:

        key = item["text"][:120]

        if key in seen:
            continue

        seen.add(key)

        results.append(item)

        if len(results) >= 8:
            break

    return results


# ============================================================
# 20. EXTRACT RELEVANT SENTENCES
# ============================================================

def extract_relevant_sentences(
    question,
    results
):

    if not results:
        return []

    topic = detect_topic(
        question
    )

    topic_keywords = QUESTION_TOPICS.get(
        topic,
        []
    )

    question_words = set(
        re.findall(
            r"[a-zA-Z]{4,}",
            question.lower()
        )
    )

    sentences = []

    for item in results:

        text = item["text"]

        parts = re.split(
            r"(?<=[.!?])\s+",
            text
        )

        for sentence in parts:

            sentence = sentence.strip()

            if len(sentence) < 35:
                continue

            lower = sentence.lower()

            score = 0

            for keyword in topic_keywords:

                if keyword in lower:
                    score += 4

            for word in question_words:

                if word in lower:
                    score += 2

            if score > 0:

                sentences.append(
                    (
                        score,
                        sentence,
                        item["url"]
                    )
                )

    sentences.sort(
        key=lambda x: x[0],
        reverse=True
    )

    final_sentences = []

    seen = set()

    for score, sentence, url in sentences:

        normalized = re.sub(
            r"\W+",
            " ",
            sentence.lower()
        ).strip()

        if normalized in seen:
            continue

        seen.add(normalized)

        final_sentences.append(
            (
                sentence,
                url
            )
        )

        if len(final_sentences) >= 8:
            break

    return final_sentences


# ============================================================
# 21. CLEAN ANSWER
# ============================================================

def clean_answer_sentence(sentence):

    sentence = sentence.strip()

    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    )

    unwanted_phrases = [

        "skip to main content",
        "click here to apply",
        "read more",
        "learn more",
        "cookie policy",
        "privacy policy",
        "terms and conditions",
        "follow us on",
        "all rights reserved",
        "contact us",
        "contact our team",
        "request info",
        "application form"

    ]

    lower_sentence = sentence.lower()

    for phrase in unwanted_phrases:

        if phrase in lower_sentence:
            return ""

    return sentence


# ============================================================
# 22. GENERATE ANSWER
# ============================================================

def generate_answer(
    question,
    results
):

    if not results:

        return """
<div class="answer-not-found">

<h2>❓ Information Not Found</h2>

<p>
I could not find relevant information for this
question on the MITT website.
</p>

<p>
Please try another suggested question.
</p>

</div>
"""

    relevant_sentences = extract_relevant_sentences(
        question,
        results
    )

    if not relevant_sentences:

        return """
<div class="answer-not-found">

<h2>❓ Information Not Found</h2>

<p>
I could not find a clear answer to this question
on the MITT website.
</p>

</div>
"""

    answer_lines = []

    used_urls = []

    for sentence, url in relevant_sentences:

        sentence = clean_answer_sentence(
            sentence
        )

        if not sentence:
            continue

        if len(sentence) > 450:

            sentence = sentence[:447] + "..."

        answer_lines.append(
            sentence
        )

        if url not in used_urls:
            used_urls.append(url)

        if len(answer_lines) >= 5:
            break

    if not answer_lines:

        return """
<div class="answer-not-found">

<h2>❓ Information Not Found</h2>

<p>
I could not find a clear answer on the MITT website.
</p>

</div>
"""

    answer = """
<div class="answer-content">

<h2>🎓 Answer</h2>

"""

    for line in answer_lines:

        answer += (
            "<div class='answer-item'>"
            + "✓ "
            + line
            + "</div>"
        )

    if used_urls:

        answer += """

<div class="source-box">

<h3>🌐 Source</h3>

<a href="{0}" target="_blank">
{0}
</a>

</div>
""".format(
            used_urls[0]
        )

    answer += "</div>"

    return answer


# ============================================================
# 23. ASK QUESTION
# ============================================================

def ask_question(question):

    question = str(
        question or ""
    ).strip()

    if not question:

        return """
<div class="answer-not-found">

<h2>❓ Select a Question</h2>

<p>
Please select a suggested question or type your own question.
</p>

</div>
"""

    if not website_loaded:

        return """
<div class="answer-not-found">

<h2>⚠️ Website Not Loaded</h2>

<p>
Please go to the <b>Load Website</b> page and load the
MITT website first.
</p>

</div>
"""

    print()
    print("=" * 70)
    print("QUESTION:", question)
    print("=" * 70)

    topic = detect_topic(
        question
    )

    print(
        "Detected topic:",
        topic
    )

    results = search_content(
        question
    )

    print(
        "Relevant chunks:",
        len(results)
    )

    return generate_answer(
        question,
        results
    )


# ============================================================
# 24. LOAD COLLEGE
# ============================================================

def load_college():

    global website_loaded

    website_loaded = False

    print()
    print("=" * 70)
    print("LOADING MITT WEBSITE")
    print("=" * 70)

    try:

        pages = crawl_website()

        if not pages:

            return """
<div class="status-error">

<h2>❌ Website Could Not Be Loaded</h2>

<p>
The MITT website could not be read.
</p>

<p>
Please check your internet connection and try again.
</p>

</div>
"""

        build_search_content()

        if not website_content:

            return """
<div class="status-error">

<h2>❌ No Content Found</h2>

<p>
The website opened, but readable content could not
be extracted.
</p>

</div>
"""

        website_loaded = True

        return """
<div class="status-success">

<h2>✅ MITT Website Loaded Successfully</h2>

<div class="stat-grid">

<div class="stat-card">
<div class="stat-number">{0}</div>
<div class="stat-label">Pages Loaded</div>
</div>

<div class="stat-card">
<div class="stat-number">{1}</div>
<div class="stat-label">Search Sections</div>
</div>

<div class="stat-card">
<div class="stat-number">{2}</div>
<div class="stat-label">URLs Visited</div>
</div>

</div>

<p>
<b>College:</b> {3}
</p>

<p>
<b>Website:</b>
<a href="{4}" target="_blank">{4}</a>
</p>

<p class="ready-text">
🚀 The Q&A system is ready.
</p>

</div>
""".format(
            len(college_pages),
            len(website_content),
            len(visited_urls),
            COLLEGE_NAME,
            COLLEGE_URL
        )

    except Exception as e:

        website_loaded = False

        print(
            "LOAD ERROR:",
            str(e)
        )

        return """
<div class="status-error">

<h2>❌ Loading Error</h2>

<p>
The website could not be loaded.
</p>

</div>
"""


# ============================================================
# 25. SUGGESTED QUESTIONS
# ============================================================

QUESTIONS = [

    "What courses does MITT offer?",

    "What undergraduate programs are available?",

    "What postgraduate programs are available?",

    "What engineering courses are offered?",

    "What is the admission process?",

    "How can I apply for admission?",

    "What are the eligibility requirements?",

    "What is the fee structure?",

    "What are the tuition fees?",

    "What are the placement opportunities?",

    "Which companies recruit students?",

    "What placement support does MITT provide?",

    "What facilities are available on campus?",

    "What hostel facilities are available?",

    "Does MITT have a library?",

    "What sports facilities are available?",

    "What student clubs and activities are available?",

    "What internship opportunities are available?",

    "What scholarships are available?",

    "What departments does MITT have?",

    "What research opportunities are available?",

    "Where is MITT located?",

    "What is MITT known for?",

    "What are the vision and mission of MITT?",

    "What facilities does MITT provide to students?",

    "Is transportation available for students?",

    "What are the campus facilities at MITT?",

    "What are the MBA and MCA programs?",

    "What M.Tech programs are available?",

    "What makes MITT different from other colleges?"

]


# ============================================================
# 26. SELECT QUESTION
# ============================================================

def select_question(question):

    return question


# ============================================================
# 27. PAGE HTML
# ============================================================

HOME_PAGE = """
<div class="page-content">

<div class="hero-card">

<div class="hero-overlay">

<div class="hero-badge">
🎓 MITT COLLEGE
</div>

<h1>
Maharaja Institute of Technology Thandavapura
</h1>

<p>
College Website Q&A Assistant
</p>

<div class="hero-buttons">
<span>📚 Courses</span>
<span>🎯 Admissions</span>
<span>💼 Placements</span>
<span>🏫 Campus</span>
</div>

</div>

</div>

<div class="welcome-card">

<h2>Welcome to MITT Q&A</h2>

<p>
This application helps students find information
from the official Maharaja Institute of Technology
Thandavapura website.
</p>

<div class="feature-grid">

<div class="feature">
<span>📖</span>
<h3>College Information</h3>
<p>Explore important college details.</p>
</div>

<div class="feature">
<span>🌐</span>
<h3>Website Search</h3>
<p>Load and search official website content.</p>
</div>

<div class="feature">
<span>💡</span>
<h3>Suggested Questions</h3>
<p>Quickly find common information.</p>
</div>

<div class="feature">
<span>🤖</span>
<h3>Smart Q&A</h3>
<p>Keyword-based relevant answers.</p>
</div>

</div>

</div>

</div>
"""


COLLEGE_DETAILS_PAGE = """
<div class="page-content">

<div class="page-title">
<h1>🏫 College Details</h1>
<p>Important information about MITT</p>
</div>

<div class="details-card">

<h2>Maharaja Institute of Technology Thandavapura</h2>

<div class="detail-row">
<span class="detail-icon">📍</span>
<div>
<strong>Location</strong>
<p>Thandavapura, Mysuru District, Karnataka</p>
</div>
</div>

<div class="detail-row">
<span class="detail-icon">🌐</span>
<div>
<strong>Official Website</strong>
<p>
<a href="https://mitt.edu.in/" target="_blank">
https://mitt.edu.in/
</a>
</p>
</div>
</div>

<div class="detail-row">
<span class="detail-icon">🏛️</span>
<div>
<strong>Institution</strong>
<p>Maharaja Institute of Technology Thandavapura</p>
</div>
</div>

<div class="detail-row">
<span class="detail-icon">🎓</span>
<div>
<strong>Institution Type</strong>
<p>Engineering & Management Institute</p>
</div>
</div>

</div>

<div class="info-card">

<h2>📌 About This Application</h2>

<p>
The Q&A assistant collects readable information
from the official MITT website and organizes it
into searchable sections.
</p>

</div>

</div>
"""


ABOUT_PAGE = """
<div class="page-content">

<div class="page-title">

<h1>ℹ️ About Project</h1>

<p>MITT College Website Q&A Assistant</p>

</div>

<div class="project-card">

<h2>💻 Technologies Used</h2>

<div class="tech-grid">

<div class="tech">
<h3>🐍 Python</h3>
<p>Main programming language.</p>
</div>

<div class="tech">
<h3>🌐 Requests</h3>
<p>Used to download website pages.</p>
</div>

<div class="tech">
<h3>🔎 BeautifulSoup</h3>
<p>Used to extract text and links from HTML.</p>
</div>

<div class="tech">
<h3>🎨 Gradio</h3>
<p>Used to create the interactive web interface.</p>
</div>

</div>

<h2>🧠 AI / Search Model</h2>

<p>
This project does not use a large language model.
It uses a lightweight <b>keyword-based information
retrieval system</b>.
</p>

<p>
The system detects the topic of the question,
searches website sections, scores relevant content,
and extracts the most relevant sentences.
</p>

<h2>🚫 Technologies Not Used</h2>

<ul>

<li>Qwen</li>

<li>FAISS</li>

<li>Vector embeddings</li>

<li>Vector database</li>

<li>Chatbot component</li>

</ul>

<h2>⚡ Main Advantage</h2>

<p>
The system is simple, fast and easy to run because
it does not require a large AI model or GPU.
</p>

</div>

</div>
"""


# ============================================================
# 28. CUSTOM CSS
# ============================================================

CUSTOM_CSS = f"""

/* ============================================================
   GLOBAL
   ============================================================ */

html,
body,
.gradio-container {{

    margin: 0 !important;
    padding: 0 !important;

    font-family:
        Arial,
        Helvetica,
        sans-serif !important;

    color: #ffffff !important;

    background: #050505 !important;
}}


/* ============================================================
   BACKGROUND
   ============================================================ */

.gradio-container {{

    background:
        linear-gradient(
            rgba(0,0,0,0.88),
            rgba(0,0,0,0.94)
        ),
        url("{BACKGROUND_IMAGE}");

    background-size: cover;

    background-position: center;

    background-attachment: fixed;
}}


/* ============================================================
   MAIN
   ============================================================ */

#app-container {{

    max-width: 1250px !important;

    margin:
        20px auto !important;

    padding:
        20px !important;

    background:
        rgba(8,8,8,0.96) !important;

    border:
        1px solid #292929 !important;

    border-radius:
        24px !important;

    box-shadow:
        0 20px 70px rgba(0,0,0,0.7) !important;
}}


/* ============================================================
   NAVBAR
   ============================================================ */

#navbar {{

    display: flex !important;

    align-items: center !important;

    justify-content: space-between !important;

    background:
        linear-gradient(
            135deg,
            #111111,
            #1c1c1c
        ) !important;

    border:
        1px solid #333333 !important;

    border-radius:
        18px !important;

    padding:
        15px 20px !important;

    margin-bottom:
        20px !important;
}}


.nav-title {{

    color: #ffffff !important;

    font-size: 21px !important;

    font-weight: 800 !important;
}}


/* ============================================================
   NAV BUTTONS
   ============================================================ */

.nav-button {{

    background:
        #151515 !important;

    color:
        #ffffff !important;

    border:
        1px solid #444444 !important;

    border-radius:
        10px !important;

    font-weight:
        700 !important;

    min-height:
        42px !important;
}}


.nav-button:hover {{

    background:
        #292929 !important;

    border-color:
        #00d4ff !important;

    color:
        #ffffff !important;
}}


/* ============================================================
   ALL MARKDOWN TEXT
   ============================================================ */

.markdown,
.markdown *,
label,
p,
h1,
h2,
h3,
h4,
h5,
h6,
span,
li,
strong {{

    color: #ffffff !important;
}}


/* ============================================================
   HERO
   ============================================================ */

.hero-card {{

    min-height:
        430px;

    border-radius:
        22px;

    overflow:
        hidden;

    background:
        linear-gradient(
            rgba(0,0,0,0.45),
            rgba(0,0,0,0.85)
        ),
        url("{BACKGROUND_IMAGE}");

    background-size:
        cover;

    background-position:
        center;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    border:
        1px solid #333333;
}}


.hero-overlay {{

    text-align:
        center;

    padding:
        50px;

    max-width:
        900px;
}}


.hero-badge {{

    display:
        inline-block;

    padding:
        8px 18px;

    border-radius:
        30px;

    background:
        #00d4ff;

    color:
        #000000 !important;

    font-weight:
        900;

    margin-bottom:
        18px;
}}


.hero-card h1 {{

    color:
        #ffffff !important;

    font-size:
        42px;

    line-height:
        1.15;

    margin:
        10px 0;
}}


.hero-card p {{

    color:
        #dddddd !important;

    font-size:
        19px;
}}


.hero-buttons {{

    display:
        flex;

    justify-content:
        center;

    flex-wrap:
        wrap;

    gap:
        10px;

    margin-top:
        25px;
}}


.hero-buttons span {{

    padding:
        10px 15px;

    background:
        rgba(255,255,255,0.10);

    border:
        1px solid rgba(255,255,255,0.25);

    border-radius:
        10px;

    color:
        #ffffff !important;
}}


/* ============================================================
   PAGE CONTENT
   ============================================================ */

.page-content {{

    padding:
        10px;
}}


.page-title {{

    background:
        linear-gradient(
            135deg,
            #151515,
            #222222
        );

    border:
        1px solid #333333;

    border-radius:
        18px;

    padding:
        25px;

    margin-bottom:
        20px;

    text-align:
        center;
}}


.page-title h1 {{

    color:
        #ffffff !important;

    font-size:
        30px;
}}


/* ============================================================
   CARDS
   ============================================================ */

.welcome-card,
.details-card,
.info-card,
.project-card {{

    background:
        linear-gradient(
            135deg,
            #111111,
            #191919
        );

    border:
        1px solid #333333;

    border-radius:
        18px;

    padding:
        25px;

    margin-top:
        20px;

    box-shadow:
        0 10px 30px rgba(0,0,0,0.35);
}}


.welcome-card h2,
.details-card h2,
.info-card h2,
.project-card h2 {{

    color:
        #00d4ff !important;
}}


/* ============================================================
   FEATURE GRID
   ============================================================ */

.feature-grid,
.tech-grid {{

    display:
        grid;

    grid-template-columns:
        repeat(4, 1fr);

    gap:
        15px;

    margin-top:
        20px;
}}


.feature,
.tech {{

    background:
        #0d0d0d;

    border:
        1px solid #303030;

    border-radius:
        15px;

    padding:
        20px;

    text-align:
        center;

    transition:
        0.2s;
}}


.feature:hover,
.tech:hover {{

    transform:
        translateY(-3px);

    border-color:
        #00d4ff;

    box-shadow:
        0 8px 25px rgba(0,212,255,0.15);
}}


.feature span {{

    font-size:
        30px;
}}


/* ============================================================
   DETAILS
   ============================================================ */

.detail-row {{

    display:
        flex;

    gap:
        18px;

    padding:
        18px 0;

    border-bottom:
        1px solid #292929;
}}


.detail-icon {{

    font-size:
        28px;
}}


.detail-row strong {{

    color:
        #00d4ff !important;
}}


/* ============================================================
   LINKS
   ============================================================ */

a {{

    color:
        #00d4ff !important;

    text-decoration:
        none !important;
}}


a:hover {{

    text-decoration:
        underline !important;
}}


/* ============================================================
   LOAD PAGE
   ============================================================ */

.load-card {{

    background:
        #111111;

    border:
        1px solid #333333;

    border-radius:
        18px;

    padding:
        30px;

    text-align:
        center;
}}


/* ============================================================
   LOAD BUTTON
   ============================================================ */

#load-button {{

    display:
        block !important;

    visibility:
        visible !important;

    opacity:
        1 !important;

    background:
        linear-gradient(
            135deg,
            #00d4ff,
            #0077ff
        ) !important;

    color:
        #000000 !important;

    border:
        none !important;

    border-radius:
        13px !important;

    min-height:
        55px !important;

    font-size:
        18px !important;

    font-weight:
        900 !important;

    margin-top:
        15px !important;
}}


#load-button:hover {{

    transform:
        translateY(-2px);

    box-shadow:
        0 10px 30px rgba(0,212,255,0.30) !important;
}}


/* ============================================================
   STATUS
   ============================================================ */

#status-box {{

    margin-top:
        20px !important;

    background:
        #0d0d0d !important;

    border:
        1px solid #333333 !important;

    border-radius:
        15px !important;

    padding:
        20px !important;
}}


.status-success h2 {{

    color:
        #00ff88 !important;
}}


.status-error h2 {{

    color:
        #ff4d4d !important;
}}


/* ============================================================
   STAT GRID
   ============================================================ */

.stat-grid {{

    display:
        grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap:
        15px;

    margin:
        20px 0;
}}


.stat-card {{

    background:
        #151515;

    border:
        1px solid #333333;

    border-radius:
        14px;

    padding:
        18px;

    text-align:
        center;
}}


.stat-number {{

    color:
        #00d4ff;

    font-size:
        30px;

    font-weight:
        900;
}}


.stat-label {{

    color:
        #aaaaaa;
}}


.ready-text {{

    color:
        #00ff88 !important;

    font-weight:
        800;
}}


/* ============================================================
   QUESTION BUTTONS
   ============================================================ */

.question-button {{

    background:
        #111111 !important;

    color:
        #ffffff !important;

    border:
        1px solid #444444 !important;

    border-radius:
        12px !important;

    min-height:
        62px !important;

    font-weight:
        700 !important;

    transition:
        0.2s !important;
}}


.question-button:hover {{

    background:
        #222222 !important;

    border-color:
        #00d4ff !important;

    color:
        #00d4ff !important;

    transform:
        translateY(-2px) !important;
}}


/* ============================================================
   TEXTBOX
   ============================================================ */

textarea,
input {{

    background:
        #111111 !important;

    color:
        #ffffff !important;

    border:
        1px solid #444444 !important;

    border-radius:
        12px !important;
}}


textarea:focus,
input:focus {{

    border-color:
        #00d4ff !important;

    box-shadow:
        0 0 0 2px rgba(0,212,255,0.15) !important;
}}


/* ============================================================
   ASK BUTTON
   ============================================================ */

#ask-button {{

    background:
        linear-gradient(
            135deg,
            #00ff88,
            #00a85a
        ) !important;

    color:
        #000000 !important;

    border:
        none !important;

    border-radius:
        13px !important;

    min-height:
        52px !important;

    font-weight:
        900 !important;

    font-size:
        17px !important;
}}


/* ============================================================
   ANSWER
   ============================================================ */

#answer-box {{

    background:
        #0d0d0d !important;

    border:
        1px solid #333333 !important;

    border-radius:
        18px !important;

    padding:
        25px !important;

    min-height:
        220px !important;
}}


.answer-content h2 {{

    color:
        #00d4ff !important;
}}


.answer-item {{

    background:
        #151515;

    border-left:
        4px solid #00d4ff;

    border-radius:
        8px;

    padding:
        14px;

    margin:
        10px 0;

    color:
        #ffffff !important;

    line-height:
        1.6;
}}


.answer-not-found h2 {{

    color:
        #ffcc00 !important;
}}


.source-box {{

    margin-top:
        25px;

    padding:
        18px;

    background:
        #151515;

    border:
        1px solid #333333;

    border-radius:
        12px;
}}


.source-box h3 {{

    color:
        #00ff88 !important;
}}


/* ============================================================
   FOOTER
   ============================================================ */

.footer {{

    text-align:
        center;

    padding:
        20px;

    margin-top:
        20px;

    border-top:
        1px solid #333333;

    color:
        #888888 !important;
}}


/* ============================================================
   MOBILE
   ============================================================ */

@media (max-width: 800px) {{

    #app-container {{

        margin:
            8px !important;

        padding:
            10px !important;
    }}

    .hero-card {{

        min-height:
            360px;
    }}

    .hero-card h1 {{

        font-size:
            28px;
    }}

    .feature-grid,
    .tech-grid {{

        grid-template-columns:
            1fr 1fr;
    }}

    .stat-grid {{

        grid-template-columns:
            1fr;
    }}

}}

@media (max-width: 500px) {{

    .feature-grid,
    .tech-grid {{

        grid-template-columns:
            1fr;
    }}

    .hero-overlay {{

        padding:
            25px;
    }}

}}


"""


# ============================================================
# 29. PAGE FUNCTIONS
# ============================================================

def show_home():

    return HOME_PAGE


def show_college_details():

    return COLLEGE_DETAILS_PAGE


def show_load_page():

    return LOAD_PAGE


def show_questions_page():

    return QUESTIONS_PAGE


def show_ask_page():

    return ASK_PAGE


def show_about_page():

    return ABOUT_PAGE


# ============================================================
# 30. LOAD PAGE
# ============================================================

LOAD_PAGE = """
<div class="page-content">

<div class="page-title">

<h1>🌐 Load Official Website</h1>

<p>
Load MITT website content before asking questions.
</p>

</div>

<div class="load-card">

<h2>🚀 Website Crawler</h2>

<p>
Click the button below to crawl the official
MITT website.
</p>

<p>
The crawler will collect readable HTML content
from pages belonging to the MITT domain.
</p>

</div>

</div>
"""


# ============================================================
# 31. QUESTIONS PAGE
# ============================================================

QUESTIONS_PAGE = """
<div class="page-content">

<div class="page-title">

<h1>💡 Suggested Questions</h1>

<p>
Choose a question and then go to Ask Question.
</p>

</div>

</div>
"""


# ============================================================
# 32. ASK PAGE
# ============================================================

ASK_PAGE = """
<div class="page-content">

<div class="page-title">

<h1>🤖 Ask MITT</h1>

<p>
Ask questions about the official MITT website.
</p>

</div>

</div>
"""


# ============================================================
# 33. GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    title="MITT Thandavapura Q&A",
    css=CUSTOM_CSS
) as demo:

    # ========================================================
    # MAIN CONTAINER
    # ========================================================

    with gr.Column(
        elem_id="app-container"
    ):

        # ====================================================
        # NAVIGATION
        # ====================================================

        with gr.Row(
            elem_id="navbar"
        ):

            gr.Markdown(
                "🎓 MITT Q&A",
                elem_classes="nav-title"
            )

            home_btn = gr.Button(
                "🏠 Home",
                elem_classes="nav-button"
            )

            details_btn = gr.Button(
                "🏫 College",
                elem_classes="nav-button"
            )

            load_page_btn = gr.Button(
                "🌐 Load Website",
                elem_classes="nav-button"
            )

            questions_page_btn = gr.Button(
                "💡 Questions",
                elem_classes="nav-button"
            )

            ask_page_btn = gr.Button(
                "🤖 Ask",
                elem_classes="nav-button"
            )

            about_btn = gr.Button(
                "ℹ️ About",
                elem_classes="nav-button"
            )


        # ====================================================
        # PAGE DISPLAY
        # ====================================================

        page_display = gr.HTML(
            value=HOME_PAGE
        )


        # ====================================================
        # LOAD WEBSITE AREA
        # ====================================================

        load_area = gr.Column(
            visible=False
        )

        with load_area:

            gr.Markdown(
                """
<div class="page-content">

<div class="page-title">

<h1>🌐 Load Official Website</h1>

<p>
Load and index information from the official MITT website.
</p>

</div>

</div>
"""
            )

            gr.Markdown(
                """
<div class="load-card">

<h2>🏛️ Maharaja Institute of Technology Thandavapura</h2>

<p>
Official Website:
<a href="https://mitt.edu.in/" target="_blank">
https://mitt.edu.in/
</a>
</p>

<p>
Click the button below to start crawling.
</p>

</div>
"""
            )

            load_button = gr.Button(
                "🚀 LOAD MITT WEBSITE",
                variant="primary",
                elem_id="load-button"
            )

            status_box = gr.HTML(
                """
<div class="status-box">

<h2>⚪ Website Not Loaded</h2>

<p>
Click <b>LOAD MITT WEBSITE</b> to start.
</p>

</div>
""",
                elem_id="status-box"
            )


        # ====================================================
        # QUESTION AREA
        # ====================================================

        question_area = gr.Column(
            visible=False
        )

        with question_area:

            gr.Markdown(
                """
<div class="page-content">

<div class="page-title">

<h1>💡 Suggested Questions</h1>

<p>
Choose a question below.
</p>

</div>

</div>
"""
            )

            question_buttons = []

            for i in range(
                0,
                len(QUESTIONS),
                3
            ):

                with gr.Row():

                    for j in range(3):

                        index = i + j

                        if index < len(QUESTIONS):

                            button = gr.Button(
                                QUESTIONS[index],
                                elem_classes="question-button"
                            )

                            question_buttons.append(
                                button
                            )


        # ====================================================
        # ASK AREA
        # ====================================================

        ask_area = gr.Column(
            visible=False
        )

        with ask_area:

            gr.Markdown(
                """
<div class="page-content">

<div class="page-title">

<h1>🤖 Ask MITT</h1>

<p>
Search the information collected from the official website.
</p>

</div>

</div>
"""
            )

            question_box = gr.Textbox(

                label="Your Question",

                placeholder=(
                    "Example: What courses does MITT offer?"
                ),

                lines=3
            )

            ask_button = gr.Button(
                "📤 GET ANSWER",
                variant="primary",
                elem_id="ask-button"
            )

            answer_box = gr.HTML(
                """
<div class="answer-not-found">

<h2>💬 Answer</h2>

<p>
Select a suggested question or type your own question.
</p>

</div>
""",
                elem_id="answer-box"
            )


        # ====================================================
        # FOOTER
        # ====================================================

        gr.HTML(
            """
<div class="footer">

🎓 MITT Thandavapura Q&A Assistant

<br>

Information is retrieved from the official MITT website.

</div>
"""
        )


    # ========================================================
    # NAVIGATION FUNCTIONS
    # ========================================================

    def hide_all():

        return (
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False)
        )


    # ========================================================
    # HOME
    # ========================================================

    def navigate_home():

        return (
            HOME_PAGE,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False)
        )


    # ========================================================
    # COLLEGE DETAILS
    # ========================================================

    def navigate_details():

        return (
            COLLEGE_DETAILS_PAGE,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False)
        )


    # ========================================================
    # LOAD PAGE
    # ========================================================

    def navigate_load():

        return (
            "",
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False)
        )


    # ========================================================
    # QUESTIONS PAGE
    # ========================================================

    def navigate_questions():

        return (
            "",
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(visible=False)
        )


    # ========================================================
    # ASK PAGE
    # ========================================================

    def navigate_ask():

        return (
            "",
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True)
        )


    # ========================================================
    # ABOUT
    # ========================================================

    def navigate_about():

        return (
            ABOUT_PAGE,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False)
        )


    # ========================================================
    # NAVIGATION EVENTS
    # ========================================================

    home_btn.click(
        fn=navigate_home,
        inputs=[],
        outputs=[
            page_display,
            load_area,
            question_area,
            ask_area
        ]
    )

    details_btn.click(
        fn=navigate_details,
        inputs=[],
        outputs=[
            page_display,
            load_area,
            question_area,
            ask_area
        ]
    )

    load_page_btn.click(
        fn=navigate_load,
        inputs=[],
        outputs=[
            page_display,
            load_area,
            question_area,
            ask_area
        ]
    )

    questions_page_btn.click(
        fn=navigate_questions,
        inputs=[],
        outputs=[
            page_display,
            load_area,
            question_area,
            ask_area
        ]
    )

    ask_page_btn.click(
        fn=navigate_ask,
        inputs=[],
        outputs=[
            page_display,
            load_area,
            question_area,
            ask_area
        ]
    )

    about_btn.click(
        fn=navigate_about,
        inputs=[],
        outputs=[
            page_display,
            load_area,
            question_area,
            ask_area
        ]
    )


    # ========================================================
    # LOAD WEBSITE
    # ========================================================

    load_button.click(

        fn=load_college,

        inputs=[],

        outputs=status_box
    )


    # ========================================================
    # QUESTION BUTTONS
    # ========================================================

    for button in question_buttons:

        button.click(

            fn=select_question,

            inputs=button,

            outputs=question_box
        )

        # Automatically open Ask page

        button.click(

            fn=navigate_ask,

            inputs=[],

            outputs=[
                page_display,
                load_area,
                question_area,
                ask_area
            ]
        )


    # ========================================================
    # ASK QUESTION
    # ========================================================

    ask_button.click(

        fn=ask_question,

        inputs=question_box,

        outputs=answer_box
    )


# ============================================================
# 34. LAUNCH
# ============================================================

print()
print("=" * 70)
print("STARTING MITT THANDAVAPURA Q&A ASSISTANT")
print("=" * 70)

demo.launch(
    share=True,
    debug=True
)


/tmp/ipykernel_949/1116551703.py:2919: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(



STARTING MITT THANDAVAPURA Q&A ASSISTANT
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0c3a8588c4fe71e9a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



LOADING MITT WEBSITE

STARTING MITT WEBSITE CRAWLER

[1/25] https://mitt.edu.in/
[200] https://mitt.edu.in/
   ✓ Saved: Home - Maharaja Institute of Technology, Thandavapura

[2/25] https://mitt.edu.in/grievance
[200] https://mitt.edu.in/grievance
   ✓ Saved: Grievance - Maharaja Institute of Technology, Thandavapura

[3/25] https://mitt.edu.in/online-payments
[200] https://mitt.edu.in/online-payments
   ✓ Saved: Online Payments - Maharaja Institute of Technology, Thandavapura

[4/25] https://mitt.edu.in/funded-project-faculty
[200] https://mitt.edu.in/funded-project-faculty
   ✓ Saved: Funded Project - Faculty - Maharaja Institute of Technology, Thandavapura

[5/25] https://mitt.edu.in/nirf
[200] https://mitt.edu.in/nirf
   ✓ Saved: NIRF - Maharaja Institute of Technology, Thandavapura

[6/25] https://mittup.mitt.edu.in/
[200] https://mittup.mitt.edu.in/
   ✓ Saved: MittUp — Campus Social Network for Students & Faculty

[7/25] https://mitt.edu.in/internal-hackathon
[200] https://mitt